# LAB 03 [SOLUTION] — Delta Lake Fundamentals: Lifecycle Management

**Databricks Fundamentals | ~80 min | Difficulty ⭐⭐⭐**

---

## Scenario

You maintain the **product catalog** for RetailHub.
New products arrive weekly, prices change, discontinued items must be removed,
and any accidental data corruption must be recoverable.

You will implement the full **Delta Lake lifecycle**:
- Create a table with constraints and a generated column
- Run DML (INSERT / UPDATE / DELETE / MERGE)
- Audit every change via `DESCRIBE HISTORY`
- Recover from a disaster using Time Travel + RESTORE
- Clean up storage with VACUUM and understand the trade-off

## Choose your path

Each TODO task below gives you **two cells to choose from** — run exactly one:

| Cell label | Language | Style |
|---|---|---|
| `[PySpark]` | Python | `spark.sql()` / DataFrame API |
| `[SQL]` | SQL | `%sql` magic — pure SQL, no Python |

The auto-checks at the end of each task work regardless of which path you took.

## Task map

| # | Title | Type | Difficulty |
|---|-------|------|-----------|
| 01 | Create table with constraints | GIVEN | ⭐ |
| 02 | Schema enforcement in action | GIVEN + ? | ⭐⭐ |
| 03 | INSERT a product batch | **TODO** | ⭐ |
| 04 | UPDATE: price change + restock | **TODO** | ⭐ |
| 05 | DELETE: retire discontinued products | **TODO** | ⭐ |
| 06 | DESCRIBE DETAIL + HISTORY audit | **TODO** | ⭐ |
| 07 | MERGE INTO — weekly delta upsert | **TODO** | ⭐⭐⭐ |
| 08 | Schema evolution: add `supplier` column | **TODO** | ⭐⭐ |
| 09 | Time Travel — VERSION AS OF | **TODO** | ⭐⭐ |
| 10 | Time Travel — TIMESTAMP AS OF + diff | **TODO** | ⭐⭐ |
| 11 | Simulate disaster → RESTORE | **TODO** | ⭐⭐ |
| 12 | VACUUM — cleanup & impact on Time Travel | **TODO** | ⭐⭐ |


In [ ]:
# Setup — run first
%run ../setup/00_setup

TABLE = f"{CATALOG}.silver.lab_products"
print(f"Working table: {TABLE}")

In [ ]:
# Set default catalog + schema so SQL cells can reference 'lab_products' directly
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("USE SCHEMA silver")
print(f"Default context set → {CATALOG}.silver")
print("SQL cells in this lab can reference the table as just:  lab_products")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 01 — Create product catalog table with constraints           [GIVEN]
# ─────────────────────────────────────────────────────────────────────────
# Delta supports:
#   NOT NULL       — enforced on every write
#   CHECK(expr)    — custom row-level predicate
#   DEFAULT value  — applied when column is omitted in INSERT
#   GENERATED AS   — computed from other columns, NEVER inserted manually

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        product_id    STRING  NOT NULL,
        name          STRING  NOT NULL,
        category      STRING,
        unit_price    DOUBLE  NOT NULL,
        stock_qty     INT     DEFAULT 0,
        is_active     BOOLEAN DEFAULT TRUE,
        created_date  DATE,
        price_band    STRING  GENERATED ALWAYS AS (
                          CASE
                              WHEN unit_price < 10  THEN 'budget'
                              WHEN unit_price < 50  THEN 'mid'
                              WHEN unit_price < 100 THEN 'premium'
                              ELSE 'luxury'
                          END
                      ),
        CONSTRAINT chk_price CHECK (unit_price > 0),
        CONSTRAINT chk_stock CHECK (stock_qty  >= 0)
    )
    USING DELTA
""")

spark.sql(f"""
    INSERT INTO {TABLE}
        (product_id, name, category, unit_price, stock_qty, is_active, created_date)
    VALUES
        ('P001', 'Laptop Pro 15',    'Electronics', 1299.99, 50, TRUE, '2024-01-05'),
        ('P002', 'Wireless Mouse',   'Electronics',   29.99,200, TRUE, '2024-01-05'),
        ('P003', 'Office Chair',     'Furniture',    349.00, 30, TRUE, '2024-01-10'),
        ('P004', 'USB-C Hub 7-in-1', 'Electronics',   49.99,150, TRUE, '2024-01-15'),
        ('P005', 'Standing Desk',    'Furniture',    599.00, 20, TRUE, '2024-02-01')
""")

spark.table(TABLE).show(truncate=False)
print(" Table created — this is version 0")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 02 — Schema enforcement in action                       [GIVEN + ?]
# ─────────────────────────────────────────────────────────────────────────
# Delta REJECTS writes that violate schema or constraints.

from datetime import date
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, BooleanType, DateType
)

# Attempt A — numeric column supplied as STRING
print("=== Attempt A: unit_price as STRING (wrong type) ===")
try:
    bad_schema = StructType([
        StructField("product_id",   StringType(),  False),
        StructField("name",         StringType(),  False),
        StructField("category",     StringType(),  True),
        StructField("unit_price",   StringType(),  False),   # ← deliberately wrong
        StructField("stock_qty",    IntegerType(), True),
        StructField("is_active",    BooleanType(), True),
        StructField("created_date", DateType(),    True),
    ])
    bad_df = spark.createDataFrame(
        [Row("P099", "Bad Product", "Test", "NOT_A_NUMBER", 10, True, date(2024, 6, 1))],
        bad_schema
    )
    bad_df.write.format("delta").mode("append").saveAsTable(TABLE)
    print("  Written — unexpected success!")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {str(e)[:200]}")

# Attempt B — CHECK constraint violation (negative price)
print("\n=== Attempt B: negative unit_price (CHECK constraint) ===")
try:
    spark.sql(f"""
        INSERT INTO {TABLE} (product_id, name, unit_price, created_date)
        VALUES ('P099', 'Ghost', -5.0, '2024-06-01')
    """)
    print("  Written — unexpected success!")
except Exception as e:
    print(f"   Caught {type(e).__name__}: {str(e)[:200]}")

# ? Which layer (Spark engine vs Delta) rejects Attempt A vs Attempt B?
# ? How is a GENERATED column different from a regular withColumn()?
# ? What happens to price_band when you UPDATE unit_price?

### Task 03 — INSERT a new product batch  `[TODO]` · *PySpark or SQL*

A new product drop arrives. Insert the 3 rows below — this creates **version 2**.

> **Important:** `price_band` is a `GENERATED ALWAYS AS` column.
> What happens if you try to supply it manually in the INSERT?

| product_id | name | category | unit_price | stock_qty | is_active | created_date |
|---|---|---|---|---|---|---|
| P006 | Noise-Cancelling Headphones | Electronics | 199.99 | 75 | TRUE | 2024-03-01 |
| P007 | Ergonomic Keyboard | Electronics | 89.00 | 120 | TRUE | 2024-03-01 |
| P008 | Monitor 27" 4K | Electronics | 449.00 | 40 | TRUE | 2024-03-05 |

**▶ PySpark path:** complete the cell marked `[PySpark path]` below  
**▶ SQL path:** complete the cell marked `[SQL path]` below — run it *instead* of the PySpark cell


In [ ]:
# Task 03 [SOLUTION] — INSERT 3 new products
spark.sql(f"""
    INSERT INTO {TABLE} (product_id,name,category,unit_price,stock_qty,is_active,created_date)
    VALUES
        ('P006','Noise-Cancelling Headphones','Electronics',199.99,75,TRUE,'2024-03-01'),
        ('P007','Ergonomic Keyboard',          'Electronics', 89.00,120,TRUE,'2024-03-01'),
        ('P008','Monitor 27in 4K',             'Electronics',449.00, 40,TRUE,'2024-03-05')
""")
print(f'Row count after INSERT: {spark.table(TABLE).count()}')
spark.table(TABLE).select('product_id','name','unit_price','price_band').show(truncate=False)

In [ ]:
%sql
-- [SQL path] Task 03 — INSERT 3 new products
-- Run this cell INSTEAD of the PySpark cell above (not both)

-- YOUR SQL HERE
-- hint: INSERT INTO lab_products (col1, col2, ...) VALUES (...), (...), (...)
-- Do NOT include price_band — it is generated automatically


In [ ]:
#  Check — Task 03: INSERT new products
_t03 = spark.table(TABLE)
assert _t03.count() == 8, f"Expected 8 rows after INSERT (5 original + 3 new), got {_t03.count()}"

# Confirm new product IDs were inserted
new_ids = {r.product_id for r in _t03.select("product_id").collect()}
for pid in ["P006", "P007", "P008"]:
    assert pid in new_ids, f"Product {pid} not found in table — INSERT not completed?"

# price_band must be auto-assigned (no NULLs allowed for generated column)
null_band = _t03.filter("price_band IS NULL").count()
assert null_band == 0, f"{null_band} rows have NULL price_band — generated column issue?"

print(f" Task 03 passed — {_t03.count()} rows, P006/P007/P008 present, price_band auto-generated")

### Task 04 — UPDATE: price change + restock  `[TODO]` · *PySpark or SQL*

Write **two separate UPDATE statements** (each creates a new Delta version):

- **Event A:** `Laptop Pro 15` (P001) price drops **10%**
- **Event B:** `Standing Desk` (P005) restocked by **+50 units**

> After updating P001's price, inspect `price_band` — did it change automatically? Why?

**▶ PySpark path:** complete the cell marked `[PySpark path]` below  
**▶ SQL path:** complete the cell marked `[SQL path]` below — run it *instead* of the PySpark cell


In [ ]:
# Task 04 [SOLUTION] — UPDATE price and stock
# Event A: 10% price drop
spark.sql(f"UPDATE {TABLE} SET unit_price=ROUND(unit_price*0.9,3) WHERE product_id='P001'")
# Event B: restock P005 +50
spark.sql(f"UPDATE {TABLE} SET stock_qty=stock_qty+50 WHERE product_id='P005'")
spark.table(TABLE).select('product_id','name','unit_price','stock_qty','price_band').show(truncate=False)

In [ ]:
%sql
-- [SQL path] Task 04 — UPDATE price and stock
-- Run this cell INSTEAD of the PySpark cell above (not both)
-- Write two separate statements separated by a semicolon, or run two %sql cells

-- Event A: 10% price drop on P001
-- YOUR SQL HERE ;

-- Event B: restock P005 by +50 units
-- YOUR SQL HERE


In [ ]:
#  Check — Task 04: UPDATE price and stock
from pyspark.sql import functions as F

_t04 = spark.table(TABLE)

# P001 price must be 10% lower than original 1299.99 → 1169.991
_p001_price = _t04.filter("product_id = 'P001'").select("unit_price").collect()[0]["unit_price"]
assert _p001_price < 1299.99, f"P001 unit_price still {_p001_price} — UPDATE not applied?"
assert abs(_p001_price - 1169.991) < 0.01, f"P001 price expected ~1169.99, got {_p001_price}"

# P005 stock must be > original 20 (restocked +50 => 70)
_p005_stock = _t04.filter("product_id = 'P005'").select("stock_qty").collect()[0]["stock_qty"]
assert _p005_stock > 20, f"P005 stock_qty still {_p005_stock} — restock UPDATE not applied?"
assert _p005_stock == 70, f"P005 stock_qty expected 70 (20+50), got {_p005_stock}"

print(f" Task 04 passed — P001 price={_p001_price:.3f} (was 1299.99) | P005 stock={_p005_stock} (was 20)")

### Task 05 — DELETE: retire discontinued products  `[TODO]` · *PySpark or SQL*

**Step 5a (given):** Mark P004 and P007 as retired — run as-is.

**Step 5b (TODO):** Delete all rows where `is_active = FALSE AND stock_qty = 0`.

> Why two steps? The `is_active` flag is set by a business process (possibly days
> before the cleanup job runs). Separating the flag from DELETE keeps both events
> in the audit log independently.

**▶ PySpark path:** complete the `-- YOUR SQL HERE` block inside the `[PySpark path]` cell  
**▶ SQL path:** complete the cell marked `[SQL path]` below — run it *instead* of the TODO block


In [ ]:
# Task 05a — mark P004 and P007 as retired (GIVEN — run as-is, both paths)
spark.sql(f"UPDATE {TABLE} SET is_active = FALSE, stock_qty = 0 WHERE product_id IN ('P004','P007')")
print("After marking retired:")
spark.table(TABLE).select("product_id", "name", "is_active", "stock_qty").show(truncate=False)

# [PySpark path] Task 05b — TODO: DELETE retired zero-stock products
spark.sql(f"""
    -- YOUR SQL HERE
""")

print(f"\nRow count after DELETE: {spark.table(TABLE).count()}")
spark.table(TABLE).show(truncate=False)


In [ ]:
%sql
-- [SQL path] Task 05b — DELETE retired zero-stock products
-- Run this cell INSTEAD of the TODO block inside the PySpark cell above

-- YOUR SQL HERE


In [ ]:
#  Check — Task 05: DELETE retired products
_t05 = spark.table(TABLE)
_count = _t05.count()

assert _count == 6, f"Expected 6 rows after DELETE (8 - 2 retired), got {_count}"

# P004 and P007 must be gone
remaining_ids = {r.product_id for r in _t05.select("product_id").collect()}
assert "P004" not in remaining_ids, "P004 still in table — DELETE not applied?"
assert "P007" not in remaining_ids, "P007 still in table — DELETE not applied?"

# No is_active=FALSE + stock_qty=0 rows should remain
leftover = _t05.filter("is_active = FALSE AND stock_qty = 0").count()
assert leftover == 0, f"{leftover} retired zero-stock rows still present"

print(f" Task 05 passed — {_count} rows remain, P004 and P007 deleted")

### Task 06 — Audit: DESCRIBE DETAIL + DESCRIBE HISTORY  `[TODO]` · *PySpark or SQL*

Delta records **every write** in the transaction log (`_delta_log/`).

Use the two metadata commands covered in M04 to answer:

1. How many physical files does the table currently consist of, and how large is it?
2. How many Delta versions exist right now — does the count match your mental model?
3. In the metrics for the DELETE operation, how many rows were actually removed?

**▶ PySpark path:** complete the `[PySpark path]` cell below  
**▶ SQL path:** run the two `[SQL path]` cells below — they use `DESCRIBE DETAIL` and `DESCRIBE HISTORY` directly


In [ ]:
# [PySpark path] Task 06 — TODO: audit queries

# 6a — physical metadata
print("=== DESCRIBE DETAIL ===")
# YOUR CODE HERE

# 6b — full history
print("\n=== DESCRIBE HISTORY ===")
# YOUR CODE HERE

# ? How many Delta versions exist right now?
# ? What is numDeletedRows in the DELETE operation metrics?
# ? Why might numFiles be larger than the number of active rows?


In [ ]:
%sql
-- [SQL path] Task 06a — physical metadata
-- Run this cell INSTEAD of the PySpark cell above
DESCRIBE DETAIL lab_products


In [ ]:
%sql
-- [SQL path] Task 06b — full operation history
-- ? How many versions? What does numDeletedRows show for the DELETE operation?
DESCRIBE HISTORY lab_products


In [ ]:
#  Check — Task 06: History audit
_history = spark.sql(f"DESCRIBE HISTORY {TABLE}")
_detail  = spark.sql(f"DESCRIBE DETAIL {TABLE}").collect()[0]

assert _history.count() >= 5, \
    f"Expected at least 5 Delta versions by now (CREATE, INSERT, INSERT, 2xUPDATE, UPDATE, DELETE), got {_history.count()}"

# Should have at least one DELETE operation logged
ops = {r["operation"] for r in _history.select("operation").collect()}
assert "DELETE" in ops, "No DELETE operation found in history — was Task 05b completed?"
assert "UPDATE" in ops, "No UPDATE operation found in history — were Tasks 04 completed?"

print(f" Task 06 passed — {_history.count()} versions in history | numFiles={_detail['numFiles']}")

### Task 07 — MERGE INTO: weekly delta upsert  `[TODO]` · *PySpark or SQL* ⭐⭐⭐

Every Monday a *weekly delta file* arrives with updates and new rows.

Write a single `MERGE INTO` statement that:
- **Updates** matching products (join on `product_id`) — apply all changed columns
- **Inserts** products that don't exist yet in the target

> Remember: `price_band` is generated — do **not** include it in SET or INSERT.

Source data is already loaded for you below as `weekly_src` (PySpark path) or as inline VALUES (SQL path).

| product_id | change |
|---|---|
| P001 | price → 1099.99, stock → 55 |
| P003 | price → 329.00, stock → 35 |
| P009 | Smart Watch Series 3 (NEW) — 249.99, qty 80 |
| P010 | Laptop Stand Aluminium (NEW) — 39.99, qty 200 |

**▶ PySpark path:** complete the `[PySpark path]` cell below  
**▶ SQL path:** complete the `[SQL path]` cell below — source data is embedded as inline VALUES


In [ ]:
# [PySpark path] Task 07 — TODO: MERGE INTO

from datetime import date
from pyspark.sql import Row

weekly_src = spark.createDataFrame([
    Row(product_id="P001", name="Laptop Pro 15",          category="Electronics",
        unit_price=1099.99, stock_qty=55,  is_active=True, created_date=date(2024,1,5)),
    Row(product_id="P003", name="Office Chair",           category="Furniture",
        unit_price=329.00,  stock_qty=35,  is_active=True, created_date=date(2024,1,10)),
    Row(product_id="P009", name="Smart Watch Series 3",   category="Wearables",
        unit_price=249.99,  stock_qty=80,  is_active=True, created_date=date(2024,4,1)),
    Row(product_id="P010", name="Laptop Stand Aluminium", category="Accessories",
        unit_price=39.99,   stock_qty=200, is_active=True, created_date=date(2024,4,1)),
])
weekly_src.createOrReplaceTempView("weekly_src")

spark.sql(f"""
    -- YOUR SQL HERE
""")

print("After MERGE:")
spark.table(TABLE).orderBy("product_id").show(truncate=False)

spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 1") \
    .select("version", "operation", "operationMetrics").show(truncate=False)
# ? How many rows were updated? How many were inserted?


In [ ]:
%sql
-- [SQL path] Task 07 — MERGE INTO using inline VALUES source
-- Run this cell INSTEAD of the PySpark cell above
--
-- Build the MERGE using a VALUES(...) subquery as the source.
-- Structure: MERGE INTO lab_products tgt USING (<values-table>) src ON tgt.product_id = src.product_id
-- Column order for VALUES: (product_id, name, category, unit_price, stock_qty, is_active, created_date)
--
-- Data to merge (from the task description above):
--   P001: update → price 1099.99, stock 55
--   P003: update → price 329.00,  stock 35
--   P009: new   → Smart Watch Series 3, Wearables, 249.99, qty 80,  date 2024-04-01
--   P010: new   → Laptop Stand Aluminium, Accessories, 39.99, qty 200, date 2024-04-01
--
-- YOUR SQL HERE


In [ ]:
#  Check — Task 07: MERGE INTO
_t07 = spark.table(TABLE)
ids  = {r.product_id for r in _t07.select("product_id").collect()}

# New products must have been inserted
assert "P009" in ids, "P009 not found — WHEN NOT MATCHED THEN INSERT missing or not applied?"
assert "P010" in ids, "P010 not found — WHEN NOT MATCHED THEN INSERT missing or not applied?"

# P001 must have the updated price (1099.99)
_p001 = _t07.filter("product_id = 'P001'").select("unit_price").collect()[0]["unit_price"]
assert abs(_p001 - 1099.99) < 0.01, \
    f"P001 price expected 1099.99 (MERGE update), got {_p001:.2f}"

# P003 must have the updated price (329.00)
_p003 = _t07.filter("product_id = 'P003'").select("unit_price").collect()[0]["unit_price"]
assert abs(_p003 - 329.00) < 0.01, \
    f"P003 price expected 329.00 (MERGE update), got {_p003:.2f}"

# Last operation should be MERGE
_last_op = spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 1").collect()[0]["operation"]
assert "MERGE" in _last_op.upper(), f"Last operation is '{_last_op}', expected MERGE"

print(f" Task 07 passed — MERGE applied | P001={_p001:.2f}, P003={_p003:.2f} | P009 and P010 inserted")

### Task 08 — Schema evolution: add `supplier` column  `[TODO]` · *PySpark or SQL*

The procurement team needs a `supplier` (STRING, nullable) column.

Use the two-step production pattern:

1. **DDL** — add the column to the existing table without rewriting any data
2. **Backfill** — populate `supplier` for **at least two** categories using UPDATE

> Existing rows will carry `NULL` until explicitly backfilled.  
> After step 1, inspect `printSchema()` — what changed and what didn't?

**▶ PySpark path:** complete the `[PySpark path]` cell below  
**▶ SQL path:** complete the `[SQL path]` cell below


In [ ]:
# [PySpark path] Task 08 — TODO: add supplier column + backfill

# Step 1 — ADD COLUMN
spark.sql(f"""
    -- YOUR SQL HERE
""")

print("Schema after ALTER:")
spark.table(TABLE).printSchema()

# Step 2 — backfill at least two categories
spark.sql(f"""
    -- YOUR SQL HERE
""")

spark.sql(f"""
    -- YOUR SQL HERE
""")

spark.table(TABLE).select("product_id", "name", "category", "supplier").show(truncate=False)


In [ ]:
%sql
-- [SQL path] Task 08 — ADD COLUMN + backfill
-- Run this cell INSTEAD of the PySpark cell above

-- Step 1: add the column
-- YOUR SQL HERE ;

-- Step 2: backfill Electronics
-- YOUR SQL HERE ;

-- Step 3: backfill at least one more category
-- YOUR SQL HERE


In [ ]:
#  Check — Task 08: ADD COLUMN supplier + backfill
_t08 = spark.table(TABLE)

assert "supplier" in _t08.columns, \
    "Column 'supplier' not found — ALTER TABLE ADD COLUMN not completed?"

# Electronics rows must have a supplier (backfilled)
_elec_no_supplier = _t08.filter("category = 'Electronics' AND supplier IS NULL").count()
assert _elec_no_supplier == 0, \
    f"{_elec_no_supplier} Electronics rows still have NULL supplier — backfill UPDATE missing?"

# At least one non-Electronics category must also be backfilled
_non_elec_filled = _t08.filter("category != 'Electronics' AND supplier IS NOT NULL").count()
assert _non_elec_filled > 0, \
    "No non-Electronics rows have a supplier — did you backfill at least one more category?"

print(f" Task 08 passed — 'supplier' column added | Electronics backfilled | {_non_elec_filled} other row(s) backfilled")

### Task 09 — Time Travel: VERSION AS OF  `[TODO]` · *PySpark or SQL*

Use `VERSION AS OF` to read the table **as it was at version 0**.

Once you have both the past snapshot and the current table, answer two business questions:

1. Which products from the initial seed **no longer exist** in the current table?
2. Which products have a **different `unit_price`** now compared to version 0?

> Think about which join type / SQL set operator gives you "rows in A but not in B".

**▶ PySpark path:** complete the `[PySpark path]` cell — set `df_v0` and write the DataFrame comparisons  
**▶ SQL path:** complete the three `[SQL path]` cells below, then run the bridge cell to set `df_v0` for the check


In [ ]:
# [PySpark path] Task 09 — TODO: time travel VERSION AS OF

from pyspark.sql import functions as F

# 9a — TODO: read version 0 into df_v0
df_v0 = None  # replace with your code

df_current = spark.table(TABLE)

# 9b — TODO: products deleted since v0 (in v0 but NOT in current)
print("Products deleted since version 0:")
# YOUR CODE HERE

# 9c — TODO: products whose unit_price changed
print("\nProducts with changed unit_price:")
# YOUR CODE HERE


In [ ]:
%sql
-- [SQL path] Task 09a — read version 0 snapshot
-- Run this cell INSTEAD of the PySpark cell above

SELECT * FROM lab_products VERSION AS OF 0


In [ ]:
%sql
-- [SQL path] Task 09b — products deleted since version 0
-- YOUR SQL HERE
-- Hint: think EXCEPT or a LEFT JOIN ... WHERE ... IS NULL



In [ ]:
%sql
-- [SQL path] Task 09c — products whose unit_price changed
-- YOUR SQL HERE
-- Hint: join lab_products VERSION AS OF 0 with current lab_products on product_id,
--       then filter where prices differ



In [ ]:
# [SQL path bridge] — sets df_v0 so the auto-check below can run
# Run this ONLY if you used the SQL path above (not the PySpark path)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(TABLE)


In [ ]:
#  Check — Task 09: Time Travel VERSION AS OF 0
assert df_v0 is not None, "df_v0 is None — did you read VERSION AS OF 0?"
assert df_v0.count() == 5, f"Version 0 should have 5 rows (initial seed), got {df_v0.count()}"

# Version 0 must NOT have the 'supplier' column (added later in Task 08)
assert "supplier" not in df_v0.columns, \
    "Version 0 should not have 'supplier' column — time travel may not be pointing to v0"

# P006, P007, P008 must NOT be in v0 (they were added in Task 03)
v0_ids = {r.product_id for r in df_v0.select("product_id").collect()}
for pid in ["P006","P007","P008"]:
    assert pid not in v0_ids, f"{pid} found in version 0 snapshot — unexpected"

print(f" Task 09 passed — v0 has {df_v0.count()} rows, no 'supplier' column, no P006/P007/P008")

### Task 10 — Time Travel: TIMESTAMP AS OF + diff  `[TODO]` · *PySpark or SQL*

Sometimes you know *when* something happened, not the version number.
Use `DESCRIBE HISTORY` to extract the **timestamp of version 1**, then read the table at that point.

Once you have `df_at_v1`:
1. Compare row counts between v1 and current
2. Compare the column lists — spot the schema difference
3. Explain why `df_at_v1` is missing the `supplier` column

**▶ PySpark path:** complete the `[PySpark path]` cell below — use `spark.read.format("delta").option(...)`  
**▶ SQL path:** run the two `[SQL path]` cells below (DESCRIBE HISTORY → copy timestamp → TIMESTAMP AS OF query), then run the bridge cell


In [ ]:
# [PySpark path] Task 10 — TODO: timestamp-based time travel + schema diff

history = spark.sql(f"DESCRIBE HISTORY {TABLE}")

# 10a — TODO: extract version 1 timestamp from history
v1_ts = None  # replace
print(f"Version 1 timestamp: {v1_ts}")

# 10b — TODO: read the table at v1_ts
df_at_v1 = None  # replace

# 10c — compare row counts
if df_at_v1 is not None:
    print(f"Version 1 rows : {df_at_v1.count()}")
print(f"Current rows   : {spark.table(TABLE).count()}")

# 10d — compare column lists
if df_at_v1 is not None:
    print(f"Version 1 columns : {df_at_v1.columns}")
print(f"Current columns   : {spark.table(TABLE).columns}")

# ? Does v1 have the 'supplier' column?
# ? What does this tell you about schema evolution + time travel?
# ? How can this break downstream pipelines reading historical snapshots?


In [ ]:
%sql
-- [SQL path] Task 10a — find version 1 timestamp from history
DESCRIBE HISTORY lab_products


In [ ]:
%sql
-- [SQL path] Task 10b — read table at version 1 timestamp
-- Replace <v1_timestamp> with the value you found in DESCRIBE HISTORY above
-- Format example: '2024-03-15T10:23:45.000+0000'

SELECT * FROM lab_products TIMESTAMP AS OF '<v1_timestamp>'


In [ ]:
# [SQL path bridge] — sets v1_ts and df_at_v1 so the auto-check below can run
# Run this ONLY if you used the SQL path above (not the PySpark path)
v1_ts = spark.sql(f"DESCRIBE HISTORY {TABLE}").filter("version = 1").select("timestamp").collect()[0][0]
df_at_v1 = spark.read.format("delta").option("timestampAsOf", str(v1_ts)).table(TABLE)
print(f"v1_ts = {v1_ts} | df_at_v1 has {df_at_v1.count()} rows | columns: {df_at_v1.columns}")


In [ ]:
#  Check — Task 10: Timestamp-based time travel
assert v1_ts is not None, "v1_ts is None — did you extract the version 1 timestamp from history?"
assert df_at_v1 is not None, "df_at_v1 is None — did you read the table at v1_ts?"

# Version 1 should have 5 rows (the initial INSERT committed in version 1)
assert df_at_v1.count() == 5, \
    f"Version 1 snapshot should have 5 rows (initial product seed), got {df_at_v1.count()}"

# Schema at v1 must NOT include 'supplier' (added in Task 08)
assert "supplier" not in df_at_v1.columns, \
    "df_at_v1 should not have 'supplier' column — it was added after version 1"

print(f" Task 10 passed — v1_ts={v1_ts} | df_at_v1 has {df_at_v1.count()} rows | no 'supplier' in v1 schema")

### Task 11 — Simulate disaster → RESTORE  `[TODO]` · *PySpark or SQL*

A junior engineer accidentally deletes all `Electronics` products.
Detect the damage and roll the table back.

Steps:
1. The accidental DELETE is already written — run it and observe
2. Use the history to figure out which version is the last *good* one
3. Write the RESTORE statement and execute it
4. Confirm the data is back and check what a new history entry looks like

> Key insight: RESTORE creates a **new** version — history is never rewritten.
> The `RESTORE TABLE ... TO VERSION AS OF` syntax uses an integer version number.

**▶ PySpark path:** complete the `-- YOUR RESTORE SQL HERE` block inside the `[PySpark path]` cell, set `last_good_version`  
**▶ SQL path:** set `last_good_version` in the bridge cell, then run the `[SQL path]` RESTORE cell


In [ ]:
# Task 11 — RESTORE from accidental DELETE
# Run this cell on BOTH paths — it handles the GIVEN step and shows DESCRIBE HISTORY

# Step 1 — accidental DELETE (GIVEN — run as-is)
print("=== BEFORE accidental DELETE ===")
print(f"Electronics count: {spark.table(TABLE).filter('category = \"Electronics\"').count()}")

spark.sql(f"DELETE FROM {TABLE} WHERE category = 'Electronics'")

print("\n=== AFTER accidental DELETE ===")
print(f"Total rows remaining: {spark.table(TABLE).count()}")
spark.table(TABLE).show(truncate=False)

# Step 2 — inspect history to identify the last good version
print("\n=== DESCRIBE HISTORY ===")
spark.sql(f"DESCRIBE HISTORY {TABLE}") \
    .select("version", "operation", "timestamp").show(truncate=False)

# [PySpark path] Step 3 — set the version and restore
last_good_version = None  # TODO: replace with the correct integer

if last_good_version is not None:
    # YOUR RESTORE SQL HERE — spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {last_good_version}")
    pass

# Step 4 — verify (both paths)
print("\n=== After RESTORE ===")
print(f"Total rows: {spark.table(TABLE).count()}")
spark.table(TABLE).show(truncate=False)

spark.sql(f"DESCRIBE HISTORY {TABLE}") \
    .select("version", "operation").show(truncate=False)


In [ ]:
# [SQL path — step A] Set last_good_version based on DESCRIBE HISTORY output above
# The check cell below needs this Python variable regardless of which path you used
last_good_version = None  # TODO: set the correct integer from DESCRIBE HISTORY


In [ ]:
%sql
-- [SQL path — step B] RESTORE the table
-- Replace <n> with the version number you set in the cell above
-- YOUR SQL HERE

-- RESTORE TABLE lab_products TO VERSION AS OF <n>


In [ ]:
#  Check — Task 11: RESTORE after accidental DELETE
assert last_good_version is not None, \
    "last_good_version is None — did you identify the last good version number?"

_t11 = spark.table(TABLE)

# Electronics rows must be back
_elec_back = _t11.filter("category = 'Electronics'").count()
assert _elec_back > 0, \
    f"No Electronics rows found after RESTORE — did you run the RESTORE statement?"

# RESTORE operation must appear in history
_ops = {r["operation"] for r in spark.sql(f"DESCRIBE HISTORY {TABLE}").select("operation").collect()}
assert "RESTORE" in _ops, "RESTORE operation not in history — was RESTORE run?"

print(f" Task 11 passed — table restored | {_t11.count()} total rows | {_elec_back} Electronics rows back")

### Task 12 — VACUUM: storage cleanup & Time Travel trade-off  `[TODO]` · *PySpark or SQL*

Every DML creates new Parquet files. Old files remain on disk for time travel.
**After VACUUM those files are gone — permanently.**

Steps:
1. `numFiles` before VACUUM is already captured — just run the given cell
2. Confirm version 0 is still readable (time travel should still work here)
3. Run `VACUUM ... RETAIN 0 HOURS` — note: this requires disabling a safety guard (`spark.databricks.delta.retentionDurationCheck.enabled`). Re-enable it immediately after.
4. Compare `numFiles` before vs after
5. Try reading version 0 again — what happens?

> Answer the three questions at the bottom of the cell.  
> **Never use `RETAIN 0 HOURS` in production.** The default retention is 7 days.

**▶ PySpark path:** complete steps 12b, 12c, 12e inside the `[PySpark path]` cell  
**▶ SQL path:** run the `[SQL path]` cells: confirm v0, disable guard (Python), VACUUM, re-enable (Python), test v0 again, bridge cell


In [ ]:
# Task 12 — VACUUM + verification

# 12a — numFiles before (GIVEN — both paths run this)
before = (spark.sql(f"DESCRIBE DETAIL {TABLE}")
               .select("numFiles", "sizeInBytes").collect()[0])
print(f"BEFORE: numFiles={before['numFiles']}, sizeInBytes={before['sizeInBytes']:,}")

# [PySpark path] 12b — confirm version 0 is still readable; print its row count
print("\nTime travel VERSION AS OF 0 BEFORE VACUUM:")
# YOUR CODE HERE

# [PySpark path] 12c — disable the safety guard, VACUUM RETAIN 0 HOURS, then re-enable
# YOUR CODE HERE

# [PySpark path] 12d — numFiles after (leave as-is; for SQL path use bridge cell instead)
after = (spark.sql(f"DESCRIBE DETAIL {TABLE}")
              .select("numFiles", "sizeInBytes").collect()[0])
print(f"\nAFTER:  numFiles={after['numFiles']}, sizeInBytes={after['sizeInBytes']:,}")
print(f"Files removed: {before['numFiles'] - after['numFiles']}")

# [PySpark path] 12e — attempt to read version 0 — handle the expected error and print it
print("\nTime travel VERSION AS OF 0 AFTER VACUUM (should fail):")
# YOUR CODE HERE

# ? What is the default VACUUM retention period and WHY is it 7 days?
# ? DESCRIBE HISTORY still shows ALL versions after VACUUM — why?
# ? In which scenario would you lower retention below 7 days?


In [ ]:
%sql
-- [SQL path] Task 12b — confirm version 0 is still readable before VACUUM
SELECT COUNT(*) AS rows_at_v0 FROM lab_products VERSION AS OF 0


In [ ]:
# [SQL path] Task 12c — disable safety guard before VACUUM, re-enable after
# (Spark config cannot be changed from %sql, so this step stays in Python)
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
print("Safety guard disabled — run the VACUUM cell now")


In [ ]:
%sql
-- [SQL path] Task 12c — run VACUUM (safety guard already disabled above)
-- YOUR SQL HERE
-- VACUUM lab_products ???


In [ ]:
# [SQL path] Task 12c — re-enable safety guard
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")
print("Safety guard re-enabled")


In [ ]:
%sql
-- [SQL path] Task 12e — attempt version 0 time travel AFTER VACUUM — expect an error
SELECT * FROM lab_products VERSION AS OF 0


In [ ]:
# [SQL path bridge] — capture numFiles AFTER VACUUM for the check cell
# Run this after the VACUUM cell above (SQL path only)
after = (spark.sql(f"DESCRIBE DETAIL {TABLE}")
              .select("numFiles", "sizeInBytes").collect()[0])
print(f"AFTER:  numFiles={after['numFiles']}, sizeInBytes={after['sizeInBytes']:,}")
print(f"Files removed: {before['numFiles'] - after['numFiles']}")


In [ ]:
#  Check — Task 12: VACUUM storage cleanup
_before_files = before["numFiles"]
_after_files  = after["numFiles"]

# After VACUUM RETAIN 0 HOURS there should be fewer files
assert _after_files <= _before_files, \
    f"numFiles after VACUUM ({_after_files}) is not less than before ({_before_files}) — was VACUUM run?"
assert (_before_files - _after_files) > 0, \
    "No files were removed by VACUUM — check that you used RETAIN 0 HOURS and disabled the guard"

# History must still show all versions (VACUUM never rewrites history)
_history_after = spark.sql(f"DESCRIBE HISTORY {TABLE}")
assert _history_after.count() >= 10, \
    f"History should still contain all versions after VACUUM, got {_history_after.count()}"

print(f" Task 12 passed — VACUUM removed {_before_files - _after_files} file(s) | history intact ({_history_after.count()} versions)")